## Minimal MS MARCO Learning-to-Rank (L2R) with Elasticsearch + XGBoost\n\nThis notebook is a **minimal working prototype** of an L2R pipeline:\n\n- BM25 retrieval via Elasticsearch (candidates)\n- Basic feature extraction\n- Train a simple Learning-to-Rank model (`xgboost.XGBRanker`)\n- Rerank the top results\n- Evaluate with **NDCG@10**\n\n### Assumptions\n- Elasticsearch is running and contains an index named **`msmarco`** with fields `pid` and `text`.\n- Files exist in the current working directory:\n  - `queries.tsv` with columns: `qid`, `query`\n  - `qrels.train.tsv` with columns: `qid`, `pid`, `relevance`\n- Elasticsearch config is stored in `.env.local`:\n  - `ES_LOCAL_URL`\n  - `ES_LOCAL_API_KEY`\n\n### Notes\n- We sample ~50 queries to keep runtime small.\n- Tokenization is intentionally simple (lowercase + whitespace split).\n

## Step 1: Setup and Imports\n\nWe import the required libraries, load Elasticsearch settings from `.env.local`, initialize the ES client, and define a few helper functions for BM25 search + simple text processing.\n

In [ ]:
import os\nimport re\nimport random\nfrom typing import Dict, Iterable, List, Tuple\n\nimport numpy as np\nimport pandas as pd\nfrom elasticsearch import Elasticsearch\nfrom sklearn.model_selection import train_test_split\nfrom xgboost import XGBRanker\n\n\ndef load_dotenv_like(path: str) -> Dict[str, str]:\n    \"\"\"Minimal .env loader (supports ${VAR} expansion).\"\"\"\n    if not os.path.exists(path):\n        raise FileNotFoundError(f\"Missing env file: {path}\")\n\n    # Start with process env to allow expansion against existing variables.\n    env = dict(os.environ)\n\n    var_ref = re.compile(r\"\\$\\{([A-Za-z_][A-Za-z0-9_]*)\\}\")\n\n    def expand(value: str) -> str:\n        def repl(m):\n            k = m.group(1)\n            return env.get(k, os.environ.get(k, \"\"))\n\n        return var_ref.sub(repl, value)\n\n    with open(path, \"r\", encoding=\"utf-8\") as f:\n        for raw in f:\n            line = raw.strip()\n            if not line or line.startswith(\"#\"):\n                continue\n            if \"=\" not in line:\n                continue\n            k, v = line.split(\"=\", 1)\n            k = k.strip()\n            v = v.strip().strip('\"').strip("'")\n            v = expand(v)\n            env[k] = v\n    return env\n\n\nENV = load_dotenv_like(\".env.local\")\n\nES_URL = ENV.get(\"ES_LOCAL_URL\")\nES_API_KEY = ENV.get(\"ES_LOCAL_API_KEY\")\nES_INDEX = ENV.get(\"ES_INDEX\", \"msmarco\")\n\nif not ES_URL:\n    raise ValueError(\"ES_LOCAL_URL not found in .env.local\")\nif not ES_API_KEY:\n    raise ValueError(\"ES_LOCAL_API_KEY not found in .env.local\")\n\nes = Elasticsearch(ES_URL, api_key=ES_API_KEY)\n\n\ndef tokenize(text: str) -> List[str]:\n    return text.lower().split()\n\n\ndef snippet(text: str, n: int = 180) -> str:\n    s = re.sub(r\"\\s+\", \" \", str(text)).strip()\n    return s[:n] + (\"…\" if len(s) > n else \"\")\n\n\ndef bm25_search(query: str, k: int = 10, index: str = None) -> List[Dict]:\n    index = index or ES_INDEX\n    q = {\"match\": {\"text\": query}}\n    try:\n        resp = es.search(index=index, query=q, size=k)\n    except TypeError:\n        # Compatibility with older clients\n        resp = es.search(index=index, body={\"query\": q}, size=k)\n\n    hits = resp.get(\"hits\", {}).get(\"hits\", [])\n    out = []\n    for h in hits:\n        src = h.get(\"_source\", {})\n        pid = src.get(\"pid\", h.get(\"_id\"))\n        out.append({\n            \"pid\": pid,\n            \"text\": src.get(\"text\", \"\"),\n            \"bm25_score\": float(h.get(\"_score\", 0.0)),\n        })\n    return out\n\n\nprint({\"ES_URL\": ES_URL, \"ES_INDEX\": ES_INDEX, \"api_key_loaded\": bool(ES_API_KEY)})\n

## Step 2: Load Data\n\nLoad `queries.tsv` and `qrels.train.tsv` into dataframes and print basic stats.\n

In [ ]:
queries = pd.read_csv(\"queries.tsv\", sep="\t", names=[\"qid\", \"query\"], dtype={\"qid\": str, \"query\": str})\nqrels = pd.read_csv(\"qrels.train.tsv\", sep="\t", names=[\"qid\", \"pid\", \"relevance\"], dtype={\"qid\": str, \"pid\": str, \"relevance\": int})\n\ndisplay(queries.head())\nprint(\"#queries rows:\", len(queries), \"unique qids:\", queries[\"qid\"].nunique())\nprint(\"#qrels rows:\", len(qrels), \"unique qids:\", qrels[\"qid\"].nunique())\nprint(\"qrels relevance distribution:\")\nprint(qrels[\"relevance\"].value_counts().sort_index())\n

## Step 3: Sample Queries\n\nTo keep this notebook fast, we sample about **50 queries** that appear in both `queries.tsv` and `qrels.train.tsv` (so we have positives).\n

In [ ]:
SEED = 42\nrandom.seed(SEED)\nnp.random.seed(SEED)\n\nqids_with_labels = set(qrels[\"qid\"].unique())\nqueries_labeled = queries[queries[\"qid\"].isin(qids_with_labels)].copy()\n\nsample_size = min(50, len(queries_labeled))\nsampled_queries = queries_labeled.sample(n=sample_size, random_state=SEED).reset_index(drop=True)\n\ndisplay(sampled_queries.head(10))\nprint(\"sampled qids:\", sampled_queries[\"qid\"].nunique())\n

## Step 4: BM25 Retrieval Demo (visual)\n\nFor a handful of queries, run BM25 retrieval from Elasticsearch and print the **top-10** results (pid + snippet + score).\n

In [ ]:
demo_n = min(5, len(sampled_queries))\ndemo_rows = sampled_queries.sample(n=demo_n, random_state=SEED)\n\nfor _, row in demo_rows.iterrows():\n    qid, qtext = row[\"qid\"], row[\"query\"]\n    print(\"\\n\" + \"=\" * 80)\n    print(f\"QID {qid}: {qtext}\")\n    hits = bm25_search(qtext, k=10)\n    for rank, h in enumerate(hits, start=1):\n        print(f\"  {rank:2d}. pid={h['pid']}  bm25={h['bm25_score']:.4f}  text=\" + snippet(h['text']))\n

## Step 5: Build Candidate Set\n\nFor each sampled query, retrieve top-100 BM25 documents and store `(qid, pid, bm25_score, text)`.\n

In [ ]:
candidate_k = 100\ncandidate_rows = []\n\nfor _, row in sampled_queries.iterrows():\n    qid, qtext = row[\"qid\"], row[\"query\"]\n    hits = bm25_search(qtext, k=candidate_k)\n    for h in hits:\n        candidate_rows.append({\n            \"qid\": qid,\n            \"query\": qtext,\n            \"pid\": str(h[\"pid\"]),\n            \"bm25_score\": float(h[\"bm25_score\"]),\n            \"text\": h[\"text\"],\n        })\n\ncandidates = pd.DataFrame(candidate_rows)\nprint(\"candidates rows:\", len(candidates), \"unique qids:\", candidates[\"qid\"].nunique(), \"unique pids:\", candidates[\"pid\"].nunique())\ndisplay(candidates.head())\n

## Step 6: Label Construction\n\nUsing qrels, assign `label = 1` if `(qid, pid)` is relevant (`relevance > 0`), else `0`. Then show class balance.\n

In [ ]:
qrels_sampled = qrels[qrels[\"qid\"].isin(sampled_queries[\"qid\"])].copy()\nrelevant_pairs = set(zip(qrels_sampled.loc[qrels_sampled[\"relevance\"] > 0, \"qid\"],\n                         qrels_sampled.loc[qrels_sampled[\"relevance\"] > 0, \"pid\"]))\n\ncandidates[\"label\"] = [1 if (qid, pid) in relevant_pairs else 0 for qid, pid in zip(candidates[\"qid\"], candidates[\"pid\"]) ]\n\nprint(\"label balance:\")\nprint(candidates[\"label\"].value_counts())\n

## Step 7: Feature Engineering (basic)\n\nFeatures per (query, doc):\n- BM25 score\n- Term overlap count\n- Query coverage (fraction of query terms present)\n- Document length (in tokens)\n

In [ ]:
# Pre-tokenize queries (once per qid)\nqid_to_qtokens = {qid: tokenize(qtext) for qid, qtext in sampled_queries[[\"qid\", \"query\"]].itertuples(index=False, name=None)}\n\ndef featurize_row(qid: str, doc_text: str, bm25_score: float) -> Tuple[float, float, float, float]:\n    q_tokens = qid_to_qtokens.get(qid, [])\n    d_tokens = tokenize(doc_text)\n    d_set = set(d_tokens)\n    overlap = sum(1 for t in q_tokens if t in d_set)\n    coverage = overlap / max(1, len(q_tokens))\n    doc_len = len(d_tokens)\n    return (float(bm25_score), float(overlap), float(coverage), float(doc_len))\n\nfeat_cols = [\"bm25_score\", \"term_overlap_count\", \"query_coverage\", \"doc_length\"]\n\nfeatures = np.vstack([\n    featurize_row(qid, txt, score)\n    for qid, txt, score in zip(candidates[\"qid\"], candidates[\"text\"], candidates[\"bm25_score\"])\n])\n\nfor i, c in enumerate(feat_cols):\n    candidates[c] = features[:, i]\n\ndisplay(candidates[[\"qid\", \"pid\"] + feat_cols + [\"label\"]].head())\n

## Step 8: Prepare Training Data\n\nRanking models require grouping by query. We sort by `qid` and build:\n- `X`: feature matrix\n- `y`: label vector\n- `group`: list of group sizes (documents per query)\n

In [ ]:
candidates_sorted = candidates.sort_values([\"qid\", \"bm25_score\"], ascending=[True, False]).reset_index(drop=True)\n\nX = candidates_sorted[feat_cols].to_numpy(dtype=np.float32)\ny = candidates_sorted[\"label\"].to_numpy(dtype=np.float32)\n\nqid_array = candidates_sorted[\"qid\"].to_numpy()\ngroup = candidates_sorted.groupby(\"qid\").size().to_list()\n\nprint(\"X shape:\", X.shape, \"y shape:\", y.shape, \"#groups:\", len(group), \"min/max group size:\", min(group), max(group))\n

## Step 9: Train L2R Model\n\nWe train an `XGBRanker`. XGBoost's ranking objectives are `rank:*` (e.g. `rank:ndcg`).\n

In [ ]:
ranker = XGBRanker(\n    objective=\"rank:ndcg\",\n    n_estimators=60,\n    learning_rate=0.1,\n    max_depth=4,\n    subsample=0.9,\n    colsample_bytree=0.9,\n    random_state=SEED,\n    n_jobs=4,\n)\n\nranker.fit(X, y, group=group)\nprint(\"Trained XGBRanker on\", len(group), \"queries and\", X.shape[0], \"(query,doc) pairs\")\n\ntry:\n    fi = ranker.feature_importances_\n    for name, val in sorted(zip(feat_cols, fi), key=lambda x: -x[1]):\n        print(f\"  {name}: {val:.4f}\")\nexcept Exception as e:\n    print(\"(feature importance unavailable)\", e)\n

## Step 10: Reranking Demo\n\nFor a few queries, compare:\n- BM25 top-10\n- L2R reranked top-10 (using model scores)\n

In [ ]:
rerank_demo_n = min(3, sampled_queries[\"qid\"].nunique())\ndemo_qids = sampled_queries[\"qid\"].drop_duplicates().sample(n=rerank_demo_n, random_state=SEED).to_list()\n\nfor qid in demo_qids:\n    qtext = sampled_queries.loc[sampled_queries[\"qid\"] == qid, \"query\"].iloc[0]\n    dfq = candidates_sorted[candidates_sorted[\"qid\"] == qid].head(10).copy()\n    Xq = dfq[feat_cols].to_numpy(dtype=np.float32)\n    dfq[\"pred_score\"] = ranker.predict(Xq)\n\n    bm25_view = dfq[[\"pid\", \"bm25_score\", \"pred_score\", \"label\", \"text\"]].copy()\n    bm25_view[\"snippet\"] = bm25_view[\"text\"].map(snippet)\n    bm25_view = bm25_view.drop(columns=[\"text\"])\n\n    reranked_view = bm25_view.sort_values(\"pred_score\", ascending=False).reset_index(drop=True)\n    bm25_view = bm25_view.reset_index(drop=True)\n\n    print(\"\\n\" + \"=\" * 80)\n    print(f\"QID {qid}: {qtext}\")\n    print(\"\\nBM25 top-10:\")\n    display(bm25_view[[\"pid\", \"bm25_score\", \"pred_score\", \"label\", \"snippet\"]])\n    print(\"\\nL2R reranked top-10:\")\n    display(reranked_view[[\"pid\", \"bm25_score\", \"pred_score\", \"label\", \"snippet\"]])\n

## Step 11: Basic Evaluation (NDCG@10)\n\nWe compute mean **NDCG@10** over the sampled queries for:\n- BM25 ranking (top-10 by BM25 score)\n- L2R ranking (rerank those same candidates by model score)\n

In [ ]:
def dcg_at_k(rels: List[float], k: int = 10) -> float:\n    rels = rels[:k]\n    if not rels:\n        return 0.0\n    return float(sum((2 ** r - 1) / np.log2(i + 2) for i, r in enumerate(rels)))\n\n\ndef ndcg_at_k(rels: List[float], k: int = 10) -> float:\n    dcg = dcg_at_k(rels, k=k)\n    ideal = dcg_at_k(sorted(rels, reverse=True), k=k)\n    return 0.0 if ideal == 0 else float(dcg / ideal)\n\n\nbm25_scores = []\nl2r_scores = []\n\nfor qid, dfq in candidates_sorted.groupby(\"qid\"):\n    top = dfq.head(10).copy()\n    rels_bm25 = top[\"label\"].astype(float).to_list()\n\n    Xq = top[feat_cols].to_numpy(dtype=np.float32)\n    preds = ranker.predict(Xq)\n    rels_l2r = top.assign(pred=preds).sort_values(\"pred\", ascending=False)[\"label\"].astype(float).to_list()\n\n    bm25_scores.append(ndcg_at_k(rels_bm25, k=10))\n    l2r_scores.append(ndcg_at_k(rels_l2r, k=10))\n\nprint(f\"Mean NDCG@10 (BM25): {np.mean(bm25_scores):.4f}\")\nprint(f\"Mean NDCG@10 (L2R rerank): {np.mean(l2r_scores):.4f}\")\n